In [1]:
import os
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

torch.cuda.empty_cache()

# model_name = "meta-llama/Llama-3.2-1B"
model_name = "Qwen/Qwen3-0.6B"
model_save_dir = "model/downloaded"
os.makedirs(model_save_dir, exist_ok=True)

# load the tokenizer and the model
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype="auto",
    device_map="cuda",
    cache_dir=model_save_dir
)


In [2]:
from pruning.utils import prepare_calibration_input
from data.datasets import get_loaders
train_data, val_data = get_loaders(name = 'harrison', nsamples=128, seed=42, seqlen=2048, tokenizer=tokenizer)

README.md: 0.00B [00:00, ?B/s]

medqa_corpus_en.py: 0.00B [00:00, ?B/s]

textbooks_en_jsonl.zip:   0%|          | 0.00/29.2M [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

Filter:   0%|          | 0/50386 [00:00<?, ? examples/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (4914897 > 131072). Running this sequence through the model will result in indexing errors


In [3]:
inps, outs, attention_mask, position_ids = prepare_calibration_input(model, train_data, device='cuda')

In [6]:
import torch
import torch.nn as nn
from pruning.utils import prepare_calibration_input
from data.datasets import get_loaders
def find_layers(module, layers=(nn.Linear,), name=""):
    """
    Recursively find the layers of certain type(s) in a module.
    Returns: dict[name -> module]
    """
    if isinstance(module, layers):
        return {name: module}
    res = {}
    for name1, child in module.named_children():
        child_name = name + "." + name1 if name != "" else name1
        res.update(find_layers(child, layers=layers, name=child_name))
    return res


class WrappedGPT:
    """
    Wrap a Linear layer to accumulate per-input-dimension squared L2 norms (row-scaler),
    used by Wanda: |W| * sqrt(E[||x||^2]) per input channel.
    """
    def __init__(self, layer, layer_id=0, layer_name="none"):
        self.layer = layer
        self.dev = self.layer.weight.device
        self.rows = layer.weight.data.shape[0]
        self.columns = layer.weight.data.shape[1]

        self.scaler_row = torch.zeros((self.columns,), device=self.dev)
        self.nsamples = 0

        self.layer_id = layer_id
        self.layer_name = layer_name

    def add_batch(self, inp, out):
        # inp: either [bsz, in_features] or [bsz, seq, in_features] depending on where hook is placed
        if inp.dim() == 2:
            inp = inp.unsqueeze(0)  # [1, bsz, in_features]
        tmp = inp.shape[0]

        if isinstance(self.layer, nn.Linear):
            if inp.dim() == 3:
                inp = inp.reshape((-1, inp.shape[-1]))  # [bsz*seq, in_features] (or [bsz, in_features])
            inp = inp.t()  # [in_features, n_tokens]

        self.scaler_row *= self.nsamples / (self.nsamples + tmp)
        self.nsamples += tmp

        inp = inp.to(torch.float32)
        self.scaler_row += (torch.norm(inp, p=2, dim=1) ** 2) / self.nsamples


def return_given_alpha(alpha, sort_res, W_metric, tmp_metric, sum_before):
    thres_cumsum = sum_before * alpha
    sort_mask = tmp_metric <= thres_cumsum.reshape((-1, 1))
    # index of last True (per row)
    idx = sort_mask.sum(dim=1, keepdims=True) - 1
    idx = torch.clamp(idx, min=0)
    thres = torch.gather(sort_res[0], dim=1, index=idx)
    W_mask = (W_metric <= thres)
    cur_sparsity = (W_mask == True).sum().float() / W_mask.numel()
    return W_mask, cur_sparsity


# ---- Qwen3-specific helpers ----
def _qwen3_get_layers(model: nn.Module):
    if hasattr(model, "model") and hasattr(model.model, "layers"):
        return model.model.layers
    raise AttributeError("Expected Qwen3-style `model.model.layers`.")

def _qwen3_make_position_ids(position_ids, seqlen, bsz, device):
    """
    Ensure position_ids is shape [bsz, seqlen] on device.
    If None, create 0..seqlen-1.
    """
    if position_ids is None:
        return torch.arange(seqlen, device=device).unsqueeze(0).expand(bsz, -1)  # [bsz, seqlen]
    position_ids = position_ids.to(device)
    if position_ids.dim() == 1:
        position_ids = position_ids.unsqueeze(0)
    if position_ids.shape[0] == 1 and bsz > 1:
        position_ids = position_ids.expand(bsz, -1)
    return position_ids

def _qwen3_compute_position_embeddings(model, hidden_states, position_ids):
    """
    Qwen3 expects `position_embeddings` as a tuple (cos, sin), produced by model.model.rotary_emb.
    """
    if not (hasattr(model, "model") and hasattr(model.model, "rotary_emb")):
        raise AttributeError("Expected Qwen3-style `model.model.rotary_emb` to compute RoPE embeddings.")
    return model.model.rotary_emb(hidden_states, position_ids)  # (cos, sin)

def _qwen3_layer_forward(model, layer, hidden_states, attention_mask=None, position_ids=None):
    """
    Forward a single Qwen3 decoder layer correctly by providing position_embeddings.
    hidden_states: [bsz, seqlen, hidden]
    Returns: hidden_states_out [bsz, seqlen, hidden]
    """
    bsz, seqlen, _ = hidden_states.shape
    dev = hidden_states.device

    pos_ids = _qwen3_make_position_ids(position_ids, seqlen, bsz, dev)
    pos_emb = _qwen3_compute_position_embeddings(model, hidden_states, pos_ids)

    out = layer(
        hidden_states,
        attention_mask=attention_mask,          # may be None; OK if your sequences are unpadded
        position_ids=pos_ids,
        position_embeddings=pos_emb,
        past_key_values=None,
        use_cache=False,
        cache_position=None,
    )
    # HF layers typically return a tuple; first element is hidden_states
    return out[0] if isinstance(out, (tuple, list)) else out


# ---- main pruning function (copy/paste) ----
@torch.no_grad()
def prune_wanda(
    sparsity_ratio,
    nsamples,
    seed,
    seqlen,
    model,
    tokenizer,
    device=torch.device("cuda:0"),
    prune_n=0,
    prune_m=0,
    use_variant=False,
):
    """
    Wanda pruning adapted for Qwen3 series.

    Requirements:
      - get_loaders(...) must exist and return a list or DataLoader of (input_ids, target_ids)
      - prepare_calibration_input(...) must exist and return (inps, outs, attention_mask, position_ids),
        where inps/outs are hidden-states entering layer 0: [nsamples, seqlen, hidden]

    Key Qwen3 fix:
      - When calling a single decoder layer directly, we must pass `position_embeddings`
        (cos,sin) computed from model.model.rotary_emb(hidden_states, position_ids).
    """
    # Cache config
    use_cache = getattr(model.config, "use_cache", False)
    model.config.use_cache = False

    model.eval()
    if device is None:
        device = next(model.parameters()).device

    print("loading calibration data")
    dataloader, _ = get_loaders(
        "c4",
        nsamples=nsamples,
        seed=seed,
        seqlen=seqlen,
        tokenizer=tokenizer,
    )
    print("dataset loading complete")

    # Collect calibration hidden-states that enter layer 0
    inps, outs, attention_mask, position_ids = prepare_calibration_input(model, dataloader, device=device)

    layers = _qwen3_get_layers(model)

    # Iterate blocks
    for i in range(len(layers)):
        layer = layers[i]
        subset = find_layers(layer, layers=(nn.Linear,))

        # Handle model parallel device map (optional)
        if hasattr(model, "hf_device_map") and f"model.layers.{i}" in model.hf_device_map:
            dev = model.hf_device_map[f"model.layers.{i}"]
            inps = inps.to(dev)
            outs = outs.to(dev)
            attention_mask = attention_mask.to(dev) if attention_mask is not None else None
            position_ids = position_ids.to(dev) if position_ids is not None else None
        else:
            dev = inps.device

        # Wrap linears to collect activation stats
        wrapped_layers = {name: WrappedGPT(mod, layer_id=i, layer_name=name) for name, mod in subset.items()}

        def add_batch(name):
            def _hook(mod, inp, out):
                # inp is a tuple; inp[0] is the tensor input to this Linear
                wrapped_layers[name].add_batch(inp[0].detach(), out.detach())
            return _hook

        handles = []
        for name, mod in subset.items():
            handles.append(mod.register_forward_hook(add_batch(name)))

        # Run layer forward nsamples times to populate wrapped_layers[name].scaler_row
        # IMPORTANT: Use Qwen3-correct layer forward (provides position_embeddings)
        for j in range(nsamples):
            hs = inps[j].unsqueeze(0)  # [1, seqlen, hidden]
            outs[j] = _qwen3_layer_forward(
                model=model,
                layer=layer,
                hidden_states=hs,
                attention_mask=attention_mask,
                position_ids=position_ids,
            ).squeeze(0)

        for h in handles:
            h.remove()

        # Compute Wanda metric and prune
        for name, mod in subset.items():
            print(f"pruning layer {i} name {name}")

            # Wanda metric: |W| * sqrt(activation_row_scaler)
            scaler = torch.sqrt(wrapped_layers[name].scaler_row.reshape((1, -1))).to(mod.weight.device)
            W_metric = torch.abs(mod.weight.data) * scaler

            W_mask = torch.zeros_like(W_metric, dtype=torch.bool)

            if prune_n != 0:
                # structured n:m sparsity per row blocks along columns
                assert prune_m > 0 and prune_n <= prune_m
                for col in range(0, W_metric.shape[1], prune_m):
                    block = W_metric[:, col:col + prune_m].float()
                    # pick n smallest in each row of the block
                    idx = torch.topk(block, prune_n, dim=1, largest=False).indices
                    W_mask.scatter_(1, col + idx, True)
            else:
                sort_res = torch.sort(W_metric, dim=-1, stable=True)

                if use_variant:
                    tmp_metric = torch.cumsum(sort_res.values, dim=1) if hasattr(sort_res, "values") else torch.cumsum(sort_res[0], dim=1)
                    sorted_vals = sort_res.values if hasattr(sort_res, "values") else sort_res[0]
                    sum_before = W_metric.sum(dim=1)

                    alpha = 0.4
                    alpha_hist = [0.0, 0.8]

                    # return_given_alpha expects sort_res as (values, indices)
                    if hasattr(sort_res, "values"):
                        sort_tuple = (sort_res.values, sort_res.indices)
                    else:
                        sort_tuple = sort_res

                    W_mask, cur_sparsity = return_given_alpha(alpha, sort_tuple, W_metric, tmp_metric, sum_before)

                    # binary search alpha to match target sparsity
                    while (torch.abs(cur_sparsity - sparsity_ratio) > 0.001) and ((alpha_hist[1] - alpha_hist[0]) >= 0.001):
                        if cur_sparsity > sparsity_ratio:
                            alpha_new = (alpha + alpha_hist[0]) / 2.0
                            alpha_hist[1] = alpha
                        else:
                            alpha_new = (alpha + alpha_hist[1]) / 2.0
                            alpha_hist[0] = alpha
                        alpha = alpha_new
                        W_mask, cur_sparsity = return_given_alpha(alpha, sort_tuple, W_metric, tmp_metric, sum_before)

                    print(f"alpha found {alpha} sparsity {cur_sparsity:.6f}")
                else:
                    # unstructured: prune fixed fraction by rank (per row)
                    idx = sort_res.indices if hasattr(sort_res, "indices") else sort_res[1]
                    k = int(W_metric.shape[1] * sparsity_ratio)
                    indices = idx[:, :k]
                    W_mask.scatter_(1, indices, True)

            # Apply pruning
            mod.weight.data[W_mask] = 0.0

        # Recompute outs after pruning for next-layer input propagation
        for j in range(nsamples):
            hs = inps[j].unsqueeze(0)
            outs[j] = _qwen3_layer_forward(
                model=model,
                layer=layer,
                hidden_states=hs,
                attention_mask=attention_mask,
                position_ids=position_ids,
            ).squeeze(0)

        # Swap buffers for next layer
        inps, outs = outs, inps

    model.config.use_cache = use_cache
    torch.cuda.empty_cache()
    return model

In [7]:
prune_wanda(sparsity_ratio=0.8, nsamples=128, seed=42, seqlen=2048, model=model, tokenizer=tokenizer, device=torch.device("cuda:0"), prune_n=0, prune_m=0, use_variant=True)

loading calibration data


Token indices sequence length is longer than the specified maximum sequence length for this model (528225 > 131072). Running this sequence through the model will result in indexing errors


dataset loading complete
pruning layer 0 name self_attn.q_proj
alpha found 0.528125 sparsity 0.799284
pruning layer 0 name self_attn.k_proj
alpha found 0.5218750000000001 sparsity 0.799711
pruning layer 0 name self_attn.v_proj
alpha found 0.5375000000000001 sparsity 0.800355
pruning layer 0 name self_attn.o_proj
alpha found 0.42500000000000004 sparsity 0.800437
pruning layer 0 name mlp.gate_proj
alpha found 0.525 sparsity 0.800642
pruning layer 0 name mlp.up_proj
alpha found 0.5406250000000001 sparsity 0.800170
pruning layer 0 name mlp.down_proj
alpha found 0.5218750000000001 sparsity 0.799662
pruning layer 1 name self_attn.q_proj
alpha found 0.5218750000000001 sparsity 0.799376
pruning layer 1 name self_attn.k_proj
alpha found 0.5125 sparsity 0.800477
pruning layer 1 name self_attn.v_proj
alpha found 0.546875 sparsity 0.799943
pruning layer 1 name self_attn.o_proj
alpha found 0.4875 sparsity 0.800280
pruning layer 1 name mlp.gate_proj
alpha found 0.4875 sparsity 0.799624
pruning layer

Qwen3ForCausalLM(
  (model): Qwen3Model(
    (embed_tokens): Embedding(151936, 1024)
    (layers): ModuleList(
      (0-27): 28 x Qwen3DecoderLayer(
        (self_attn): Qwen3Attention(
          (q_proj): Linear(in_features=1024, out_features=2048, bias=False)
          (k_proj): Linear(in_features=1024, out_features=1024, bias=False)
          (v_proj): Linear(in_features=1024, out_features=1024, bias=False)
          (o_proj): Linear(in_features=2048, out_features=1024, bias=False)
          (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
          (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
        )
        (mlp): Qwen3MLP(
          (gate_proj): Linear(in_features=1024, out_features=3072, bias=False)
          (up_proj): Linear(in_features=1024, out_features=3072, bias=False)
          (down_proj): Linear(in_features=3072, out_features=1024, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen3RMSNorm((1024,), eps=1e-06)
        (post_attention_layer

In [8]:
from pruning.utils import check_sparsity

check_sparsity(model)

layer 0 sparsity 0.800062
layer 1 sparsity 0.799705
layer 2 sparsity 0.799941
layer 3 sparsity 0.799869
layer 4 sparsity 0.799790
layer 5 sparsity 0.800071
layer 6 sparsity 0.799896
layer 7 sparsity 0.799941
layer 8 sparsity 0.800130
layer 9 sparsity 0.799370
layer 10 sparsity 0.800118
layer 11 sparsity 0.799839
layer 12 sparsity 0.800176
layer 13 sparsity 0.800253
layer 14 sparsity 0.800045
layer 15 sparsity 0.799984
layer 16 sparsity 0.800008
layer 17 sparsity 0.799960
layer 18 sparsity 0.800070
layer 19 sparsity 0.800473
layer 20 sparsity 0.800170
layer 21 sparsity 0.800341
layer 22 sparsity 0.799884
layer 23 sparsity 0.800445
layer 24 sparsity 0.800338
layer 25 sparsity 0.799955
layer 26 sparsity 0.800110
layer 27 sparsity 0.799545


0.800017515818278